# Nuclear shape analysis across area-defined size classes

Contour-based morphometry of nuclei segmented in [ilastik](https://www.ilastik.org/) and analysed with [CellTool](https://github.com/zpincus/celltool) and Python.
This notebook accompanies a master's thesis. It takes **binary masks** (several nuclei of different sizes per image are allowed) and produces

1. 2D nuclear outlines (CellTool iso-contour extraction, B-spline smoothing, resampling to a fixed number of points),
2. **size classes** assigned by projected area (default: *Micronuclei*, *Unconfined*, *Mildly confined* and *Confined nuclei*),
3. rigid alignment of outlines and **principal component analysis (PCA) of shape**, computed twice (shape only and size + shape),
4. group statistics (Kruskal–Wallis with ε² effect size, pairwise Kolmogorov–Smirnov tests),
5. publication-style figures and tables.

**How to run:** create the environment from `environment.yml`, put your masks in `data/masks/` (or set `USE_DEMO_DATA = True` to try the pipeline on synthetic nuclei), adjust the configuration cell and run all cells. Everything is written to `output/`.

> **Two things to keep in mind when interpreting the results**
>
> * `celltool align_contours` aligns outlines **rigidly** (translation, rotation, optional reflection) and does **not** normalise size. PCA on such outlines therefore mixes size and shape (typically mode 1 is nuclear size). To separate the two, PCA is computed in two variants: `size_included` (outlines as aligned) and `shape_only` (every outline rescaled to the same area before PCA).
> * Size classes are defined by **area thresholds**. Differences in area between the classes are therefore a consequence of the classification and are **not tested statistically**. They are a proxy for the degree of nuclear flattening, not an independent measurement of confinement.

## 1. Configuration

In [ ]:
import os, sys, glob, shutil, subprocess, warnings, platform
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.patches import Ellipse, Patch
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import LogLocator, FuncFormatter, NullFormatter
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)     # pkg_resources deprecation notice printed by CellTool


Edit only this cell. All paths are relative to the folder that contains the notebook.

In [ ]:
# ============================== INPUT / OUTPUT ==============================
USE_DEMO_DATA = False                     # True: generate synthetic masks (data/demo_masks) to test the pipeline
MASKS_DIR     = Path("data/demo_masks")   # binary masks (.tif/.tiff/.png)
OUT_DIR       = Path("output")
CONTOUR_DIR   = OUT_DIR / "contours"      # extracted outlines
ALIGNED_DIR   = OUT_DIR / "aligned"       # aligned outlines (per class and all together)
MODEL_DIR     = OUT_DIR / "models"        # CellTool shape models and measurements
FIG_DIR       = OUT_DIR / "figures"
TABLE_DIR     = OUT_DIR / "tables"

# ============================== CONTOUR EXTRACTION ==============================
RUN_EXTRACT   = True                      # False: reuse contours already present in CONTOUR_DIR
STRIP_PREFIX  = "Mask of "                # prefix added to file names by some ilastik exports (removed from contour names)
PIXEL_SIZE_UM = 0.0680209                 # micrometres per pixel
MIN_AREA_PX   = 30                        # discard objects smaller than this (pixels)
N_POINTS      = 100                       # points per resampled outline
SMOOTHING     = 0.001                     # CellTool --smoothing-factor (B-spline smoothing, in micrometres)

# ============================== ALIGNMENT / PCA ==============================
ALLOW_REFLECTION   = True                 # allow mirrored outlines during alignment
MAX_ITER           = 30                   # maximum alignment iterations
VARIANCE_EXPLAINED = 0.95                 # cumulative variance used to choose the number of modes (per-class models)
EXPORT_CELLTOOL_MODELS = True             # also write CellTool shape models and measurement CSVs

# ============================== SIZE CLASSES ==============================
# Integer area ranges in PIXELS (from, to), both ends inclusive. Contour areas are fractional, so a nucleus belongs to
# a class if  from <= area < to + 1.  Nuclei outside all ranges are excluded from the analysis.
SIZE_GROUPS_PX = [
    (10,    4999),
    (5000,  14999),
    (15000, 24999),
    (25000, 45000),
]
GROUP_NAMES = {                           # key = "from–to" (as built from SIZE_GROUPS_PX), value = name used in figures
    "10–4999":     "Micronuclei",
    "5000–14999":  "Unconfined nuclei",
    "15000–24999": "Mildly confined nuclei",
    "25000–45000": "Confined nuclei",
}
GROUP_COLORS = ["#7f7f7f", "#4c72b0", "#dd8452", "#e8472b"]   # one colour per class

# ============================== FIGURES ==============================
FONT_SIZE       = 14                      # base font size of the exploratory figures
MAX_MODES_SHOWN = 10                      # modes shown per class in the per-class mode figures
COMPOSITE_FONT  = 13                      # base font size of the composite figure
COMPOSITE_SIZE  = (15, 14)                # composite figure size (inches)
PCA_PANEL_A     = "shape_only"            # PCA shown in panel A: "shape_only" or "size_included"
PCA_PANEL_C     = "size_included"         # PCA shown in panel C: "shape_only" or "size_included"
ELLIPSE_LEVEL   = 0.95                    # fraction of data enclosed by the ellipses in panel C
AREA_LOG_SCALE  = False                   # log scale for the area axis (useful when classes differ by orders of magnitude)
SHOW_MODES_4_6  = True                    # extra column with modes 4-6 in panel A

## 2. Setup and helper functions

In [ ]:
for d in (OUT_DIR, CONTOUR_DIR, ALIGNED_DIR, MODEL_DIR, FIG_DIR, TABLE_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ---- size classes ----
LABELS = [f"{lo}–{hi}" for lo, hi in SIZE_GROUPS_PX]
for (lo1, hi1), (lo2, hi2) in zip(SIZE_GROUPS_PX, SIZE_GROUPS_PX[1:]):
    assert lo1 <= hi1 < lo2 <= hi2, f"Size ranges must be increasing and non-overlapping: {(lo1, hi1)}, {(lo2, hi2)}"
_colors = list(GROUP_COLORS)
if len(_colors) < len(LABELS):
    _colors += [plt.cm.plasma(x) for x in np.linspace(0.1, 0.9, len(LABELS) - len(_colors))]
COLORS = dict(zip(LABELS, _colors))
tag  = lambda label: label.replace("–", "-").replace(" ", "_")     # safe folder / file name
disp = lambda label: GROUP_NAMES.get(label, label)                 # name shown in figures
DISP_LABELS = [disp(l) for l in LABELS]
GROUP_BY_STEM = {}                                                 # contour name -> size class (filled in section 4)

plt.rcParams.update({
    "font.size": FONT_SIZE, "axes.titlesize": FONT_SIZE + 1, "axes.labelsize": FONT_SIZE + 1,
    "xtick.labelsize": FONT_SIZE - 1, "ytick.labelsize": FONT_SIZE - 1, "legend.fontsize": FONT_SIZE - 1,
    "axes.linewidth": 1.6, "xtick.major.width": 1.6, "ytick.major.width": 1.6,
    "xtick.major.size": 6, "ytick.major.size": 6,
})

def celltool(cmd, *args, show=True):
    "Run a CellTool command without a shell (file names with spaces and wildcards are safe)."
    exe = shutil.which("celltool") or str(Path(sys.executable).parent / "celltool")
    if not Path(exe).exists() and shutil.which("celltool") is None:
        raise FileNotFoundError("CellTool executable not found. Activate the conda environment from environment.yml.")
    res = subprocess.run([exe, cmd, *map(str, args)], capture_output=True, text=True)
    text = res.stdout + "\n" + res.stderr
    if res.returncode != 0:
        raise RuntimeError(f"celltool {cmd} failed:\n{text[-1500:]}")
    if show:
        for line in text.replace("\r", "\n").splitlines():
            l = line.strip()
            if l.startswith(("Warning:", "NOTE:")):
                print(f"  [{cmd}] {l}")
    return text

def group_of(name):
    return GROUP_BY_STEM.get(name)

def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"

def closed(p):
    return np.vstack([p, p[:1]])

def draw_outline(ax, p, color, lw=1.8, dx=0.0, dy=0.0):
    q = closed(p) + np.array([dx, dy])
    ax.plot(q[:, 0], q[:, 1], color=color, lw=lw, solid_capstyle="round")

def nice_bar(width):
    "Round scale-bar length (1, 2 or 5 x 10^k) of about 20 % of the given width."
    target = width * 0.2
    k = np.floor(np.log10(target))
    for m in (5, 2, 1):
        if m * 10**k <= target:
            return m * 10**k
    return 10**k

def add_scale_bar(ax, length, units="µm", fontsize=None):
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    xs = x0 + 0.05 * (x1 - x0)
    ys = y0 + 0.05 * (y1 - y0)                       # bottom of the axis (also correct for an inverted y axis)
    ax.plot([xs, xs + length * np.sign(x1 - x0)], [ys, ys], color="k", lw=3.5, solid_capstyle="butt")
    ax.text(xs + length * np.sign(x1 - x0) / 2, ys + 0.08 * (y1 - y0), f"{length:g} {units}",
            ha="center", va="center", fontsize=fontsize or FONT_SIZE - 2)

print("Working directory:", Path.cwd())

## 3. Demo data (optional)

With `USE_DEMO_DATA = True` the notebook generates synthetic binary masks (several nuclei per image, log-uniform areas, smaller objects with rougher outlines) so that the whole pipeline can be tested without real data. The synthetic nuclei carry no biological meaning.

In [ ]:
def generate_demo_masks(out_dir, n_images=40, size=1400, seed=0):
    "Synthetic binary masks: up to 8 non-overlapping nuclei per image."
    from skimage import io, draw
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    th = np.linspace(0, 2 * np.pi, 200, endpoint=False)
    for i in range(n_images):
        img = np.zeros((size, size), np.uint8); placed = []
        for _ in range(400):
            if len(placed) >= 8:
                break
            area_px = np.exp(rng.uniform(np.log(150), np.log(40000)))
            elong = rng.uniform(1.0, 1.7)
            a = np.sqrt(area_px * elong / np.pi); b = a / elong
            cx, cy = rng.uniform(1.4 * a, size - 1.4 * a, 2)
            if any(np.hypot(cx - x, cy - y) < 1.3 * (a + r) for x, y, r in placed):
                continue
            placed.append((cx, cy, a))
            rough = 0.10 / (1 + area_px / 3000)                  # small objects are rougher
            r_th = 1 + sum(rng.normal(0, rough / k) * np.cos(k * th + rng.uniform(0, 6.28)) for k in (2, 3, 4, 5))
            rot = rng.uniform(0, 2 * np.pi)
            x = a * r_th * np.cos(th); y = b * r_th * np.sin(th)
            xr = x * np.cos(rot) - y * np.sin(rot) + cx; yr = x * np.sin(rot) + y * np.cos(rot) + cy
            rr, cc = draw.polygon(yr, xr, img.shape); img[rr, cc] = 255
        io.imsave(out_dir / f"demo_{i:02d}.tif", img, check_contrast=False)
    print(f"Wrote {n_images} synthetic masks to {out_dir}")

if USE_DEMO_DATA and not any(MASKS_DIR.glob("*.tif")):
    generate_demo_masks(MASKS_DIR)

## 4. Contour extraction and size classes

`celltool extract_contours` finds closed iso-intensity contours at the mid-point intensity of each mask (`skimage.measure.find_contours`), keeps objects larger than `MIN_AREA_PX`, converts coordinates to micrometres, smooths every outline with a periodic cubic B-spline and resamples it to `N_POINTS` evenly spaced points. A mask containing several nuclei gives several contours (`<mask name>-1`, `<mask name>-2`, ...).

Each outline is then assigned to a size class by its **projected area in pixels** (`SIZE_GROUPS_PX`). The area is taken from the smoothed contour, so it differs from a pixel count by a few percent at most.

In [ ]:
from celltool import simple_interface

if RUN_EXTRACT:
    masks = sorted(glob.glob(str(MASKS_DIR / "*.tif")) + glob.glob(str(MASKS_DIR / "*.tiff")) + glob.glob(str(MASKS_DIR / "*.png")))
    if not masks:
        raise FileNotFoundError(f"No masks found in {MASKS_DIR}/. Put binary masks there or set USE_DEMO_DATA = True.")
    print(f"Masks: {len(masks)}")
    celltool("extract_contours", "-q", f"--min-area={MIN_AREA_PX}", f"--scale={PIXEL_SIZE_UM}", "--units=microns",
             f"--resample-points={N_POINTS}", f"--smoothing-factor={SMOOTHING}", f"--destination={CONTOUR_DIR}", *masks)

if STRIP_PREFIX:
    for f in CONTOUR_DIR.glob(f"{STRIP_PREFIX}*.contour"):
        f.rename(f.with_name(f.name.replace(STRIP_PREFIX, "", 1)))

contour_files = sorted(glob.glob(str(CONTOUR_DIR / "*.contour")))
raw_contours  = simple_interface.load_contours(contour_files)
stems         = [Path(f).stem for f in contour_files]
area_px_raw   = np.array([c.area() for c in raw_contours]) / PIXEL_SIZE_UM**2      # um^2 -> pixels
raw_by_stem   = dict(zip(stems, raw_contours))

def assign_by_area(a_px):
    for lab, (lo, hi) in zip(LABELS, SIZE_GROUPS_PX):
        if lo <= a_px < hi + 1:
            return lab
    return None

GROUP_BY_STEM.clear()
GROUP_BY_STEM.update({st: assign_by_area(a) for st, a in zip(stems, area_px_raw)})
group_files = {lab: [f for f, st in zip(contour_files, stems) if GROUP_BY_STEM[st] == lab] for lab in LABELS}
n_out = sum(v is None for v in GROUP_BY_STEM.values())

print(f"\nContours: {len(contour_files)}")
for lab in LABELS:
    print(f"  {disp(lab):>24} ({lab} px): {len(group_files[lab])}")
if n_out:
    print(f"  outside all size ranges: {n_out} (excluded)")

In [ ]:
# Threshold check: distribution of nucleus areas, bars coloured by size class
grp = np.array([GROUP_BY_STEM[st] for st in stems], dtype=object)
ok  = area_px_raw > 0
bins = np.logspace(np.log10(area_px_raw[ok].min()), np.log10(area_px_raw[ok].max()), 60)
parts  = [area_px_raw[ok & (grp == lab)] for lab in LABELS] + [area_px_raw[ok & (grp == None)]]
colors = [COLORS[lab] for lab in LABELS] + ["#c9ced6"]                       # grey = outside all classes

fig, ax = plt.subplots(figsize=(11, 4.8))
ax.hist(parts, bins=bins, stacked=True, color=colors, edgecolor="white", linewidth=0.5)
ax.set_xscale("log")
ax.set_xlabel("Nucleus area (pixels, log scale)"); ax.set_ylabel("Number of nuclei")
for e in sorted({lo for lo, _ in SIZE_GROUPS_PX} | {hi + 1 for _, hi in SIZE_GROUPS_PX}):
    ax.axvline(e, color="#222", lw=1.2, ls="--", alpha=0.7)
handles = [Patch(facecolor=COLORS[lab], label=f"{disp(lab)} ({lab} px), n = {len(group_files[lab])}") for lab in LABELS]
if len(parts[-1]):
    handles.append(Patch(facecolor="#c9ced6", label=f"outside all classes, n = {len(parts[-1])}"))
ax.legend(handles=handles, loc="lower left", bbox_to_anchor=(0, 1.02), ncol=2, frameon=False,
          fontsize=FONT_SIZE - 2, columnspacing=1.5)
for s_ in ("top", "right"): ax.spines[s_].set_visible(False)
plt.tight_layout()
plt.savefig(FIG_DIR / "S1_size_class_thresholds.png", dpi=200, bbox_inches="tight")
plt.show()

**Where to place the thresholds.** A dashed line avoids cutting through a population only where the distribution has a natural valley (a density minimum). Inside a single peak there is no natural boundary, so splitting it into several classes is a convention. The cell below lists density valleys and, as an alternative, splits the main peak into classes with equal numbers of nuclei. It changes nothing by itself; the printed ranges can be pasted into `SIZE_GROUPS_PX`.

In [ ]:
from scipy.stats import gaussian_kde

N_GROUPS_IN_MAIN_PEAK = 3        # number of classes in the main peak (nuclei above the deepest valley)
KDE_BANDWIDTH = 0.25             # larger = smoother density, fewer valleys

x = np.log10(area_px_raw[area_px_raw > 0])
grid = np.linspace(x.min(), x.max(), 800)
dens = gaussian_kde(x, bw_method=KDE_BANDWIDTH)(grid)
idx = np.where((dens[1:-1] < dens[:-2]) & (dens[1:-1] < dens[2:]))[0] + 1

if len(idx) == 0:
    print("No density valleys found: the area distribution looks like a single peak (try a smaller KDE_BANDWIDTH).")
else:
    print("Density valleys (natural boundaries between populations):")
    for i in idx:
        depth = dens[i] / dens.max()
        print(f"  ~{10**grid[i]:.0f} px   (density {depth:.2f} of maximum" + (", shallow: may not be a real boundary)" if depth > 0.5 else ")"))
    cut = 10 ** grid[idx[np.argmin(dens[idx])]]
    main = np.sort(area_px_raw[area_px_raw >= cut])
    qs = np.percentile(main, np.linspace(0, 100, N_GROUPS_IN_MAIN_PEAK + 1)[1:-1])
    edges = [int(round(cut))] + [int(round(q)) for q in qs]
    lo_all, hi_all = int(np.floor(area_px_raw[area_px_raw > 0].min())), int(np.ceil(main.max()))
    suggestion = [(lo_all, edges[0] - 1)] + [(a, b - 1) for a, b in zip(edges, edges[1:] + [hi_all + 1])]
    print(f"\nDeepest valley ~{cut:.0f} px. The {len(main)} nuclei above it are split into {N_GROUPS_IN_MAIN_PEAK} classes of equal size:")
    print("SIZE_GROUPS_PX = [" + ", ".join(map(str, suggestion)) + "]")

## 5. Alignment (CellTool)

Outlines are aligned by rotation and translation (and reflection if `ALLOW_REFLECTION`) with `celltool align_contours`, an iterative procedure that rotates every outline towards the current mean shape. **No scaling is applied.** If CellTool prints `did not converge`, increase `MAX_ITER`, try `ALLOW_REFLECTION = False`, and inspect the outlines for artefacts such as merged nuclei or debris.

* **Per class** (`output/aligned/<class>/`): used for the per-class mode figures; optionally a CellTool shape model per class (`output/models/Model_<class>.contour`).
* **All classes together** (`output/aligned/All/`): one common coordinate system, used for all comparisons between classes. Optionally an `AllModel` and CellTool measurement tables per class are written as well.

In [ ]:
refl = ["--allow-reflection"] if ALLOW_REFLECTION else []

# ---- per class ----
for lab in LABELS:
    t, files = tag(lab), group_files[lab]
    if len(files) < 4:
        print(f"{disp(lab)}: too few contours ({len(files)}), skipped"); continue
    dest = ALIGNED_DIR / t; dest.mkdir(parents=True, exist_ok=True)
    print(f"> {disp(lab)} (n = {len(files)})")
    celltool("align_contours", "-q", *refl, f"--max-iterations={MAX_ITER}", f"--destination={dest}", *files)
    if EXPORT_CELLTOOL_MODELS:
        celltool("shape_model", "-q", f"--variance-explained={VARIANCE_EXPLAINED}",
                 f"--output-prefix={MODEL_DIR / ('Model_' + t)}", *sorted(glob.glob(str(dest / "*.contour"))))

# ---- all classes together ----
all_files = [f for lab in LABELS for f in group_files[lab]]
dest_all = ALIGNED_DIR / "All"; dest_all.mkdir(parents=True, exist_ok=True)
print(f"> all classes together (n = {len(all_files)})")
celltool("align_contours", "-q", *refl, f"--max-iterations={MAX_ITER}", f"--destination={dest_all}", *all_files)
aligned_all = sorted(glob.glob(str(dest_all / "*.contour")))

if EXPORT_CELLTOOL_MODELS:
    from celltool.contour import contour_class
    celltool("shape_model", "-q", f"--variance-explained={VARIANCE_EXPLAINED}",
             f"--output-prefix={MODEL_DIR / 'AllModel'}", "--no-data", *aligned_all)
    n_modes = len(contour_class.from_file(str(MODEL_DIR / "AllModel.contour")).modes)
    modes = [str(m) for m in range(1, min(2, n_modes) + 1)]
    for lab in LABELS:
        fl = [f for f in aligned_all if group_of(Path(f).stem) == lab]
        celltool("measure_contours", "-q", f"--output-file={TABLE_DIR / ('celltool_measurements_' + tag(lab) + '.csv')}",
                 "--area", "--centroid", "--shape-modes", str(MODEL_DIR / "AllModel.contour"), *modes, "-", *fl)
    print(f"AllModel: {n_modes} modes at {VARIANCE_EXPLAINED:.0%} variance; modes {', '.join(modes)} measured per class")
print("Done.")

## 6. Load aligned outlines into Python

All following analyses use the jointly aligned outlines. Projected area is taken from the outlines (µm²); rigid alignment does not change it.

In [ ]:
contours = simple_interface.load_contours(aligned_all)
names  = np.array([c.simple_name() for c in contours])
groups = np.array([group_of(n) for n in names], dtype=object)
keep   = groups != None
contours = [c for c, k in zip(contours, keep) if k]
names, groups = names[keep], groups[keep]

points = np.array([c.points for c in contours])        # (N nuclei, P points, 2), micrometres
area   = np.array([c.area() for c in contours])        # um^2

df = pd.DataFrame({"contour": names, "group": groups, "area_um2": area})
df.insert(2, "group_name", df["group"].map(disp))
print("Outline array:", points.shape)
display(df.groupby("group", sort=False)["area_um2"].agg(["count", "mean", "std", "median"]).round(2).loc[LABELS])

## 7. PCA of nuclear shape: two variants

Each outline (P points, 2 coordinates) is one vector of length 2P. PCA finds orthogonal directions (shape modes) of maximal variance in this space; each mode can be visualised as the mean outline displaced by ±*k* standard deviations along it, and every nucleus receives a score per mode.

| Variant | Procedure | Describes |
|---|---|---|
| `size_included` | PCA on the aligned outlines as they are (CellTool behaviour) | size **and** shape |
| `shape_only` | every outline rescaled to the mean area before PCA | shape only |

Scores are divided by the standard deviation of their mode ("normalized s.d."). The sign of a mode is arbitrary; here it is chosen so that the correlation of the mode with area is positive.

In [ ]:
SD_COLORS = {-2: "#f2a33a", -1: "#e0622d", 0: "#9b3d96", 1: "#3b2fd0", 2: "#000000"}

def polygon_area(p):
    x, y = p[:, 0], p[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

def shape_pca(pts, areas, normalize_size=False, n_modes=6):
    X = pts.astype(float).copy()
    if normalize_size:
        a = np.array([polygon_area(p) for p in X])
        X = X * (np.sqrt(a.mean() / a))[:, None, None]          # common area = mean area
    N = len(X)
    flat = X.reshape(N, -1)
    mean = flat.mean(0)
    U, S, Vt = np.linalg.svd(flat - mean, full_matrices=False)
    var = S**2 / (N - 1)
    scores = (flat - mean) @ Vt.T
    for k in range(min(n_modes, scores.shape[1])):              # sign convention: positive correlation with area
        if np.corrcoef(scores[:, k], areas)[0, 1] < 0:
            Vt[k] *= -1; scores[:, k] *= -1
    sd = np.sqrt(var)
    return dict(mean=mean.reshape(-1, 2), modes=Vt[:n_modes].reshape(n_modes, -1, 2), sd=sd[:n_modes],
                ratio=(var / var.sum())[:n_modes], scores=scores[:, :n_modes], norm=scores[:, :n_modes] / sd[:n_modes])

def shape_at(res, k, t):
    "Outline at t standard deviations along mode k (0-based)."
    return res["mean"] + t * res["sd"][k] * res["modes"][k]

results = {"size_included": shape_pca(points, area, normalize_size=False),
           "shape_only":    shape_pca(points, area, normalize_size=True)}

rows = []
for variant, res in results.items():
    for k in range(6):
        rows.append({"variant": variant, "mode": k + 1, "variance_explained_%": 100 * res["ratio"][k],
                     "spearman_rho_with_area": stats.spearmanr(res["scores"][:, k], area).correlation})
pca_table = pd.DataFrame(rows)
pca_table.to_csv(TABLE_DIR / "pca_variance_and_area_correlation.csv", index=False)
display(pca_table.round(3).set_index(["variant", "mode"]))

**Reading the table:** if mode 1 of `size_included` has |ρ| close to 1, it describes nuclear size. In `shape_only` the correlation with area should drop; what remains is shape.

## 8. Statistics

For every mode the four classes are compared with the **Kruskal–Wallis test**, with the effect size **ε² = (H − k + 1) / (n − k)** (H: test statistic, k: number of classes, n: number of nuclei). In addition, all pairs of classes are compared with two-sample **Kolmogorov–Smirnov** tests (statistic *D*, Bonferroni-corrected p).

> * Classes are defined by area, so a test on area (or on a mode that is mostly size) is circular and is **not** reported as evidence. The informative question is whether **shape** (`shape_only`) differs between classes.
> * With hundreds of nuclei almost every comparison gives p < 0.001. Look at the effect sizes (ε², *D*) and at the distributions.
> * Nuclei from the same image are not independent observations. Confirm key findings at the level of independent experiments or images.

In [ ]:
def kw_effect(values_by_group):
    "Kruskal-Wallis p-value and effect size epsilon^2 = (H - k + 1) / (n - k)."
    H, p = stats.kruskal(*values_by_group)
    n = sum(len(v) for v in values_by_group); k = len(values_by_group)
    return H, p, max(0.0, (H - k + 1) / (n - k))

def pairwise_ks(values, label):
    samples = [values[groups == g] for g in LABELS]
    pairs = list(combinations(range(len(LABELS)), 2))
    out = []
    for i, j in pairs:
        ks = stats.ks_2samp(samples[i], samples[j])
        out.append({"variable": label, "group_A": disp(LABELS[i]), "group_B": disp(LABELS[j]),
                    "KS_D": ks.statistic, "p": ks.pvalue, "p_Bonferroni": min(1.0, ks.pvalue * len(pairs))})
    return pd.DataFrame(out)

kw_rows, ks_tables = [], []
for variant, res in results.items():
    for k in range(3):
        H, p, eps = kw_effect([res["norm"][groups == g, k] for g in LABELS])
        kw_rows.append({"variant": variant, "mode": k + 1, "H": H, "p": p, "epsilon_sq": eps})
        ks_tables.append(pairwise_ks(res["norm"][:, k], f"{variant}: mode {k+1}"))

kw_df = pd.DataFrame(kw_rows); ks_df = pd.concat(ks_tables, ignore_index=True)
kw_df.to_csv(TABLE_DIR / "kruskal_wallis_modes.csv", index=False)
ks_df.to_csv(TABLE_DIR / "pairwise_ks_modes.csv", index=False)
display(kw_df.round(4))
display(ks_df[ks_df.variable.str.startswith("shape_only")].round(4))

## 9. Main figure: shape modes, outlines, area and shape space

| Panel | Content |
|---|---|
| **A** | Shape modes 1-3 (outlines at −2 … +2 s.d., boxplots of normalized scores by class, variance explained, Kruskal–Wallis p and ε²) and modes 4-6 |
| **B** | Aligned outlines of each class at a common scale, and projected area of individual nuclei |
| **C** | Nuclei in the space of modes 1 and 2 with data ellipses and class means |

Panel A uses `PCA_PANEL_A` (default: `shape_only`), panel C uses `PCA_PANEL_C` (default: `size_included`). To use the same PCA in both panels, set both parameters to the same value. Colours are shared across the figure and explained by a single legend, so the boxplots carry no x tick labels. Boxplots hide outliers and their axes are cut at the whiskers; panel C reports how many points fall outside the plotted area. Files: `output/figures/Figure_composite.{png,pdf,svg}`.

In [ ]:
def data_ellipse(ax, x, y, color, level=0.95, lw=3):
    "Ellipse enclosing the fraction `level` of the data (Gaussian approximation). Returns its extent (xmin, xmax, ymin, ymax)."
    cov = np.cov(x, y)
    vals, vecs = np.linalg.eigh(cov)
    w, h = 2 * np.sqrt(stats.chi2.ppf(level, 2) * vals[::-1])
    ang = np.arctan2(vecs[1, -1], vecs[0, -1])
    cx, cy = np.mean(x), np.mean(y)
    ax.add_patch(Ellipse((cx, cy), w, h, angle=np.degrees(ang), fill=False, ec=color, lw=lw))
    hx = np.hypot(w / 2 * np.cos(ang), h / 2 * np.sin(ang))
    hy = np.hypot(w / 2 * np.sin(ang), h / 2 * np.cos(ang))
    return cx - hx, cx + hx, cy - hy, cy + hy

PCA_DESCRIPTION = {"shape_only": "shape only, size normalized", "size_included": "size + shape"}

def make_composite_figure(pca_a=PCA_PANEL_A, pca_c=PCA_PANEL_C, font=COMPOSITE_FONT, size=COMPOSITE_SIZE,
                          show_modes_4_6=SHOW_MODES_4_6, area_log=AREA_LOG_SCALE, ellipse_level=ELLIPSE_LEVEL):
    res_a, res_c = results[pca_a], results[pca_c]
    G = len(LABELS)
    gcolors = [COLORS[g] for g in LABELS]
    rc = {"font.size": font, "axes.titlesize": font, "axes.labelsize": font,
          "xtick.labelsize": font - 1, "ytick.labelsize": font - 1, "legend.fontsize": font,
          "axes.linewidth": 1.4, "xtick.major.width": 1.4, "ytick.major.width": 1.4,
          "pdf.fonttype": 42, "svg.fonttype": "none"}                      # editable text in PDF / SVG

    def panel_label(ax, letter, dx=-0.16, dy=1.10):
        ax.text(dx, dy, letter, transform=ax.transAxes, fontsize=font + 9, fontweight="bold", va="bottom")

    def clean(ax):
        for s_ in ("top", "right"): ax.spines[s_].set_visible(False)

    def outline_axes(ax, res, k, lw, bar=True):
        "Outlines of mode k at -2 ... +2 s.d., own scale and scale bar."
        lim = np.abs(np.concatenate([shape_at(res, k, t) for t in (-2, 2)])).max() * 1.15
        for t in (-2, -1, 0, 1, 2):
            draw_outline(ax, shape_at(res, k, t), SD_COLORS[t], lw=lw)
        ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
        ax.set_aspect("equal", adjustable="datalim"); ax.axis("off")
        if bar:
            ax.apply_aspect(); add_scale_bar(ax, nice_bar(2 * lim), "µm", fontsize=font - 2)

    with plt.rc_context(rc):
        fig = plt.figure(figsize=size)
        outer = GridSpec(3, 1, figure=fig, height_ratios=[0.13, 1.0, 1.2], hspace=0.22,
                         top=0.965, bottom=0.075, left=0.07, right=0.98)

        # ---------- shared legend ----------
        lg = GridSpecFromSubplotSpec(2, 1, subplot_spec=outer[0], hspace=0.0)
        gh = [Patch(facecolor=COLORS[g], edgecolor="#333", label=disp(g)) for g in LABELS]
        sh = [plt.Line2D([0], [0], color=SD_COLORS[t], lw=3.5, label=l)
              for t, l in zip((-2, -1, 0, 1, 2), ("−2 s.d.", "−1 s.d.", "Mean", "+1 s.d.", "+2 s.d."))]
        for row, (title_, handles, ncol) in enumerate((("Groups", gh, G), ("Mode position", sh, 5))):
            ax_l = fig.add_subplot(lg[row]); ax_l.axis("off")
            ax_l.text(0.0, 0.5, title_, fontweight="bold", va="center", transform=ax_l.transAxes)
            ax_l.legend(handles=handles, loc="center left", bbox_to_anchor=(0.115, 0.5), ncol=ncol,
                        frameon=False, columnspacing=1.8, handlelength=1.8)

        # ---------- A: shape modes ----------
        ncolA = 4 if show_modes_4_6 else 3
        rowA = GridSpecFromSubplotSpec(1, ncolA, subplot_spec=outer[1], width_ratios=[1, 1, 1, 0.45][:ncolA], wspace=0.22)
        for k in range(3):
            ax_t = fig.add_subplot(rowA[k]); ax_t.axis("off")              # carries the shared title
            data = [res_a["norm"][groups == g, k] for g in LABELS]
            _, p, eps = kw_effect(data)
            ax_t.set_title(f"Shape mode {k+1} ({100*res_a['ratio'][k]:.1f}% of variance)\n"
                           f"KW p {fmt_p(p)}, ε² = {eps:.2f}", loc="left", fontsize=font - 1)
            if k == 0: panel_label(ax_t, "A", dx=-0.04, dy=1.12)

            cell = GridSpecFromSubplotSpec(1, 2, subplot_spec=rowA[k], width_ratios=[1.0, 1.05], wspace=0.62)
            ax_o = fig.add_subplot(cell[0]); ax_b = fig.add_subplot(cell[1])
            outline_axes(ax_o, res_a, k, lw=2.4)
            bp = ax_b.boxplot(data, positions=range(1, G + 1), widths=0.78, patch_artist=True, showfliers=False,
                              boxprops=dict(linewidth=1.6), medianprops=dict(color="white", lw=2.4),
                              whiskerprops=dict(color="#333", lw=1.6), capprops=dict(color="#333", lw=1.6))
            for patch, c in zip(bp["boxes"], gcolors):
                patch.set_facecolor(c); patch.set_edgecolor("#333")
            ends = np.concatenate([w.get_ydata() for w in bp["whiskers"]])
            pad = 0.08 * np.ptp(ends)
            ax_b.set_ylim(ends.min() - pad, ends.max() + pad)
            ax_b.set_xticks([]); ax_b.set_ylabel("Normalized s.d."); clean(ax_b)

        if show_modes_4_6:
            sub = GridSpecFromSubplotSpec(3, 1, subplot_spec=rowA[3], hspace=0.5)
            for r, k in enumerate((3, 4, 5)):
                ax = fig.add_subplot(sub[r])
                outline_axes(ax, res_a, k, lw=1.8, bar=False)
                ax.set_title(f"Mode {k+1} ({100*res_a['ratio'][k]:.1f}%)", fontsize=font - 1, fontweight="bold")

        # ---------- bottom row ----------
        bottom = GridSpecFromSubplotSpec(1, 3, subplot_spec=outer[2], width_ratios=[0.8, 0.8, 1.6], wspace=0.40)

        # B (left): aligned outlines at a common scale
        sub = GridSpecFromSubplotSpec(G, 1, subplot_spec=bottom[0], hspace=0.40)
        limB = np.abs(points).max() * 1.05
        for r, g in enumerate(LABELS):
            ax = fig.add_subplot(sub[r])
            sel = points[groups == g]
            for p_ in sel:
                q = closed(p_)
                ax.plot(q[:, 0], q[:, 1], color=COLORS[g], lw=0.9, alpha=0.4)
            ax.set_xlim(-limB, limB); ax.set_ylim(-limB, limB)
            ax.set_aspect("equal", adjustable="datalim"); ax.axis("off")
            ax.set_title(f"{disp(g)} (n = {len(sel)})", loc="left", color=COLORS[g], fontweight="bold", pad=2)
            if r == 0: panel_label(ax, "B", dx=0.0, dy=1.35)
            if r == G - 1:
                ax.apply_aspect(); add_scale_bar(ax, nice_bar(2 * limB), "µm", fontsize=font - 2)

        # B (right): projected area, no test (classes are defined by area)
        ax_c = fig.add_subplot(bottom[1])
        data = [area[groups == g] for g in LABELS]
        rng = np.random.default_rng(0)
        bp = ax_c.boxplot(data, positions=range(1, G + 1), widths=0.74, patch_artist=True, showfliers=False,
                          boxprops=dict(linewidth=1.6), medianprops=dict(color="#222", lw=2.4),
                          whiskerprops=dict(color="#333", lw=1.6), capprops=dict(color="#333", lw=1.6))
        for patch, c in zip(bp["boxes"], gcolors):
            patch.set_facecolor(c); patch.set_alpha(0.85); patch.set_edgecolor("#333")
        for i, d in enumerate(data, start=1):
            ax_c.scatter(i + rng.uniform(-0.2, 0.2, len(d)), d, s=9, color="#222", alpha=0.45, linewidths=0, zorder=3)
        if area_log:
            ax_c.set_yscale("log")
            ax_c.yaxis.set_major_locator(LogLocator(base=10, subs=(1.0, 2.0, 5.0)))
            ax_c.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))     # 0.1, 0.2, 0.5, 1, 2, 5, 10 ...
            ax_c.yaxis.set_minor_formatter(NullFormatter())
        ax_c.set_ylabel("Projected area (µm²)")
        ax_c.set_xticks([]); clean(ax_c)

        # C: shape space
        ax_d = fig.add_subplot(bottom[2])
        x, y = res_c["norm"][:, 0], res_c["norm"][:, 1]
        ell = []
        for g in LABELS:
            m = groups == g
            ax_d.scatter(x[m], y[m], s=26, color=COLORS[g], alpha=0.5, linewidths=0)
            ell.append(data_ellipse(ax_d, x[m], y[m], COLORS[g], ellipse_level))
            ax_d.scatter(x[m].mean(), y[m].mean(), marker="X", s=170, color=COLORS[g], edgecolors="k", zorder=5)
        def view(v):
            lo, hi = np.percentile(v, [0.5, 99.5]); pad = 0.12 * (hi - lo)
            return lo - pad, hi + pad
        ell = np.array(ell)
        (x0, x1), (y0, y1) = view(x), view(y)
        x0, x1 = min(x0, ell[:, 0].min()), max(x1, ell[:, 1].max())        # ellipses are never clipped
        y0, y1 = min(y0, ell[:, 2].min()), max(y1, ell[:, 3].max())
        px, py = 0.03 * (x1 - x0), 0.03 * (y1 - y0)
        x0, x1, y0, y1 = x0 - px, x1 + px, y0 - py, y1 + py
        ax_d.set_xlim(x0, x1); ax_d.set_ylim(y0, y1)
        n_out = int(((x < x0) | (x > x1) | (y < y0) | (y > y1)).sum())
        if n_out:
            ax_d.text(0.98, 0.02, f"{n_out} points outside view", transform=ax_d.transAxes,
                      ha="right", va="bottom", fontsize=font - 3, color="#555")
        ax_d.axhline(0, color="#ccc", lw=1, zorder=0); ax_d.axvline(0, color="#ccc", lw=1, zorder=0)
        ax_d.set_xlabel(f"Mode 1 ({100*res_c['ratio'][0]:.1f}%), normalized s.d.")
        ax_d.set_ylabel(f"Mode 2 ({100*res_c['ratio'][1]:.1f}%), normalized s.d.")
        ax_d.set_title(PCA_DESCRIPTION[pca_c].capitalize(), fontsize=font)
        clean(ax_d); panel_label(ax_d, "C", dx=-0.16, dy=1.04)

        fig.text(0.07, 0.985, f"Panel A: PCA, {PCA_DESCRIPTION[pca_a]}.  Panel C: PCA, {PCA_DESCRIPTION[pca_c]}.",
                 ha="left", va="bottom", fontsize=font - 1, color="#555")
        fig.text(0.07, 0.022,
                 f"Boxes: median and IQR, whiskers 1.5×IQR, outliers not shown (A, B). Ellipses (C): {int(100*ellipse_level)}% data ellipse, "
                 "X = class mean. Classes are defined by projected area, so area differences are not tested.",
                 fontsize=font - 3, color="#555", va="top")

        for ext, kw in (("png", dict(dpi=300)), ("pdf", {}), ("svg", {})):
            fig.savefig(FIG_DIR / f"Figure_composite.{ext}", bbox_inches="tight", **kw)
        plt.show()

make_composite_figure()

## 10. Supplementary views per size class

**S2. Outlines coloured by point order.** All outlines of each class; every segment is coloured by the index of the contour point (black → blue → purple → orange), which shows the start point and direction of each outline and any inconsistent alignment. *Extracted* shows the nuclei where they lie in the image, *Aligned* shows them after per-class alignment.

**S3. Shape modes per class.** Separate PCA for each class on its own aligned outlines (size not normalised): outlines at −2 … +2 s.d. for the modes that together explain `VARIANCE_EXPLAINED` of the variance (at most `MAX_MODES_SHOWN`). Because size is not normalised, mode 1 often reflects size in classes with a wide size range.

In [ ]:
CT_SD_COLORS = {-2: "#000000", -1: "#3b10e0", 0: "#9a2a86", 1: "#f26a30", 2: "#f9a23c"}      # CellTool colour scheme
CT_CMAP = LinearSegmentedColormap.from_list("celltool", [CT_SD_COLORS[k] for k in (-2, -1, 0, 1, 2)])

def draw_points_colored(ax, pts_list, lw=1.4):
    "Outlines with every segment coloured by point index."
    segs, cols = [], []
    for p in pts_list:
        q = closed(p)
        s_ = np.stack([q[:-1], q[1:]], axis=1)
        segs.append(s_); cols.append(CT_CMAP(np.linspace(0, 1, len(s_))))
    ax.add_collection(LineCollection(np.concatenate(segs), colors=np.concatenate(cols), linewidths=lw))
    allp = np.concatenate(pts_list)
    pad_x, pad_y = 0.04 * np.ptp(allp[:, 0]), 0.04 * np.ptp(allp[:, 1])
    ax.set_xlim(allp[:, 0].min() - pad_x, allp[:, 0].max() + pad_x)
    ax.set_ylim(allp[:, 1].max() + pad_y, allp[:, 1].min() - pad_y)          # y down, as in the image
    ax.set_aspect("equal"); ax.axis("off")

def figure_group_contours(source, save_stem, ncols=2):
    "source: 'extracted' (position in the image) or 'aligned' (after per-class alignment)."
    G = len(LABELS); nrows = int(np.ceil(G / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 8 * nrows))
    axes = np.atleast_1d(axes).ravel()
    for ax, g in zip(axes, LABELS):
        if source == "extracted":
            pts = [raw_by_stem[Path(f).stem].points for f in group_files[g]]
        else:
            files = sorted(glob.glob(str(ALIGNED_DIR / tag(g) / "*.contour")))
            pts = [c.points for c in simple_interface.load_contours(files)] if files else []
        if not pts:
            ax.axis("off"); ax.set_title(f"{disp(g)}: no contours"); continue
        draw_points_colored(ax, pts)
        add_scale_bar(ax, nice_bar(np.ptp(np.concatenate(pts)[:, 0])), "µm")
        ax.set_title(f"{disp(g)} (n = {len(pts)})", fontsize=FONT_SIZE + 3, fontweight="bold")
    for ax in axes[G:]:
        ax.axis("off")
    ttl = "Extracted contours" if source == "extracted" else "Aligned contours"
    fig.suptitle(f"{ttl}, coloured by point order along the contour", fontsize=FONT_SIZE + 4, y=1.0)
    plt.tight_layout()
    for ext in ("png", "svg"):
        fig.savefig(FIG_DIR / f"{save_stem}.{ext}", dpi=200, bbox_inches="tight")
    plt.show()

figure_group_contours("extracted", "S2a_outlines_by_class_extracted")
figure_group_contours("aligned", "S2b_outlines_by_class_aligned")

In [ ]:
def figure_group_modes(max_modes=MAX_MODES_SHOWN, ncols=5):
    for g in LABELS:
        files = sorted(glob.glob(str(ALIGNED_DIR / tag(g) / "*.contour")))
        if len(files) < 4:
            print(f"{disp(g)}: too few outlines for PCA ({len(files)})"); continue
        cs = simple_interface.load_contours(files)
        pts = np.array([c.points for c in cs]); ar = np.array([c.area() for c in cs])
        res = shape_pca(pts, ar, normalize_size=False, n_modes=min(len(cs) - 1, 40))
        cum = np.cumsum(res["ratio"])
        n_show = int(min(np.searchsorted(cum, VARIANCE_EXPLAINED) + 1, max_modes, len(cum)))

        lim = max(np.abs(np.concatenate([shape_at(res, k, t) for t in (-2, 2)])).max() for k in range(n_show)) * 1.1
        ncols_ = min(ncols, n_show); nrows = int(np.ceil(n_show / ncols_))
        fig, axes = plt.subplots(nrows, ncols_, figsize=(max(3.8 * ncols_, 9), 3.9 * nrows + 1.8), squeeze=False)
        for k, ax in enumerate(axes.ravel()):
            if k >= n_show:
                ax.axis("off"); continue
            for t in (-2, -1, 0, 1, 2):
                draw_outline(ax, shape_at(res, k, t), CT_SD_COLORS[t], lw=2.2)
            ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal"); ax.axis("off")
            ax.set_title(f"Shape mode {k+1}\n({100*res['ratio'][k]:.1f}% of total variance)", fontsize=FONT_SIZE - 1)
        add_scale_bar(axes[nrows - 1][0], nice_bar(2 * lim), "µm")
        handles = [plt.Line2D([0], [0], color=CT_SD_COLORS[t], lw=3.5, label=lab)
                   for t, lab in zip((-2, -1, 0, 1, 2), ("−2 s.d.", "−1 s.d.", "Mean", "+1 s.d.", "+2 s.d."))]
        fig.legend(handles=handles, loc="lower center", ncol=5, frameon=False, fontsize=FONT_SIZE - 1)
        fig.suptitle(f"{disp(g)} (n = {len(cs)}): shape modes\n"
                     f"{n_show} modes shown, {100*cum[n_show-1]:.0f}% of total variance",
                     fontsize=FONT_SIZE + 3, fontweight="bold", y=1.0)
        plt.tight_layout(rect=(0, 0.06, 1, 0.93))
        for ext in ("png", "svg"):
            fig.savefig(FIG_DIR / f"S3_modes_{tag(g)}.{ext}", dpi=200, bbox_inches="tight")
        plt.show()

figure_group_modes()

## 11. Tables

In [ ]:
for variant, res in results.items():
    for k in range(6):
        df[f"{variant}_mode{k+1}_normSD"] = res["norm"][:, k]
df.to_csv(TABLE_DIR / "all_nuclei_measurements.csv", index=False)

summary = (df.groupby("group", sort=False)
             .agg(n=("area_um2", "size"), area_mean_um2=("area_um2", "mean"),
                  area_sd_um2=("area_um2", "std"), area_median_um2=("area_um2", "median"))
             .loc[LABELS].round(2))
summary.index = [disp(l) for l in summary.index]; summary.index.name = "class"
summary.to_csv(TABLE_DIR / "summary_by_class.csv")
display(summary)

print("\nOutput files:")
for d in (TABLE_DIR, FIG_DIR):
    for f in sorted(d.glob("*")):
        print(f"  {f}")

## 12. Software versions

Printed and saved to `output/software_versions.txt` for reproducibility.

In [ ]:
import importlib.metadata as md_

def ver(pkg):
    try:
        return md_.version(pkg)
    except md_.PackageNotFoundError:
        return "not found"

lines = [f"Python {platform.python_version()} ({platform.system()} {platform.machine()})"]
lines += [f"{p}: {ver(p)}" for p in ("numpy", "scipy", "pandas", "matplotlib", "scikit-image", "tifffile", "celltool", "freeimage")]
try:
    d_ = md_.distribution("celltool")
    du = [f for f in d_.files if f.name == "direct_url.json"]
    if du:
        import json
        lines.append("celltool commit: " + json.load(open(d_.locate_file(du[0])))["vcs_info"]["commit_id"])
except Exception:
    pass
(OUT_DIR / "software_versions.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))